# Module 5 — Image Segmentation
## Industrial Defect Detection | MVTec AD Dataset

**Segmentation Strategies:**
1. Grayscale + CLAHE + Otsu  
2. Difference Image + Otsu  
3. HSV Saturation + Otsu  
4. K-Means Clustering

**Metrics:** IoU (Intersection over Union) vs ground-truth masks

## 0️⃣ Setup

In [ ]:
!git clone https://github.com/OmarAymanZaid/industrial-vision-defect-detection.git
%cd industrial-vision-defect-detection
!git pull
!pip install -r requirements.txt -q

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import sys
from tqdm import tqdm
from collections import defaultdict

sys.path.append('modules')

from segmentation import (
    segment_image,
    segment_grayscale_otsu,
    segment_diff_otsu,
    segment_hsv_saturation,
    segment_kmeans,
    compute_iou,
    evaluate_segmentation,
    batch_segment,
    visualize_segmentation,
    visualize_overlay,
    compare_strategies,
    print_segmentation_metrics
)
from utils import load_image_paths

print('✅ Segmentation module loaded')

In [ ]:
# ─── Dataset path ───────────────────────────────────────────
DATA_PATH = '/kaggle/input/datasets/ipythonx/mvtec-ad'

# Choose category (change to test another)
CATEGORY  = 'bottle'

# Discover all available categories
ALL_CATEGORIES = sorted([
    d for d in os.listdir(DATA_PATH)
    if os.path.isdir(os.path.join(DATA_PATH, d, 'train'))
])
print(f'Found {len(ALL_CATEGORIES)} categories:')
for c in ALL_CATEGORIES:
    print(f'  - {c}')

# Setup paths
test_folder  = os.path.join(DATA_PATH, CATEGORY, 'test')
gt_folder    = os.path.join(DATA_PATH, CATEGORY, 'ground_truth')
defect_types = sorted([d for d in os.listdir(test_folder) if d != 'good'])
first_defect = defect_types[0]

# Load sample images
good_img_path   = os.path.join(test_folder, 'good',
                               sorted(os.listdir(os.path.join(test_folder, 'good')))[0])
defect_img_path = os.path.join(test_folder, first_defect,
                               sorted(os.listdir(os.path.join(test_folder, first_defect)))[0])
gt_mask_path    = os.path.join(gt_folder, first_defect,
                               os.path.basename(defect_img_path).replace('.png', '_mask.png'))

img_good   = cv2.imread(good_img_path)
img_defect = cv2.imread(defect_img_path)
gt_mask    = cv2.imread(gt_mask_path, cv2.IMREAD_GRAYSCALE) if os.path.exists(gt_mask_path) else None

print(f'\nCategory     : {CATEGORY}')
print(f'Defect types : {defect_types}')
print(f'Demo defect  : {first_defect}')
print(f'GT mask      : {"loaded" if gt_mask is not None else "not found"}')

## 1️⃣ Input Images Preview

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(cv2.cvtColor(img_good,   cv2.COLOR_BGR2RGB))
axes[0].set_title('Good Sample');       axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(img_defect, cv2.COLOR_BGR2RGB))
axes[1].set_title(f'Defective ({first_defect})'); axes[1].axis('off')

if gt_mask is not None:
    axes[2].imshow(gt_mask, cmap='hot')
    axes[2].set_title('Ground Truth Mask')
else:
    axes[2].set_title('No GT Mask')
axes[2].axis('off')

plt.suptitle(f'Input Images — {CATEGORY}', fontsize=14)
plt.tight_layout()
plt.show()

## 2️⃣ Strategy 1: Grayscale + CLAHE + Otsu

In [ ]:
mask_gray = segment_grayscale_otsu(img_defect)
iou_gray  = compute_iou(mask_gray, gt_mask)

print(f'Strategy : Grayscale + CLAHE + Otsu')
print(f'IoU      : {iou_gray:.4f}' if iou_gray is not None else 'IoU: N/A')

visualize_segmentation(img_defect, mask_gray, gt_mask,
                       iou=iou_gray, strategy='Grayscale + Otsu',
                       title=f'{CATEGORY} / {first_defect}')

## 3️⃣ Strategy 2: Difference Image + Otsu

In [ ]:
mask_diff = segment_diff_otsu(img_defect, ref_good_bgr=img_good)
iou_diff  = compute_iou(mask_diff, gt_mask)

print(f'Strategy : Difference Image + Otsu')
print(f'IoU      : {iou_diff:.4f}' if iou_diff is not None else 'IoU: N/A')

# Show difference image
gray_defect = cv2.cvtColor(img_defect, cv2.COLOR_BGR2GRAY)
gray_good   = cv2.cvtColor(img_good,   cv2.COLOR_BGR2GRAY)
gray_good   = cv2.resize(gray_good, (gray_defect.shape[1], gray_defect.shape[0]))
diff_img    = cv2.absdiff(gray_defect, gray_good)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(cv2.cvtColor(img_defect, cv2.COLOR_BGR2RGB)); axes[0].set_title('Defective'); axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(img_good,   cv2.COLOR_BGR2RGB)); axes[1].set_title('Reference (Good)'); axes[1].axis('off')
axes[2].imshow(diff_img, cmap='hot');                        axes[2].set_title('Difference Image'); axes[2].axis('off')
axes[3].imshow(mask_diff, cmap='hot');                       axes[3].set_title(f'Predicted Mask | IoU: {iou_diff:.4f}' if iou_diff else 'Predicted Mask'); axes[3].axis('off')
plt.suptitle('Strategy 2: Difference + Otsu', fontsize=14)
plt.tight_layout()
plt.show()

## 4️⃣ Strategy 3: HSV Saturation + Otsu

In [ ]:
# Find a contamination-type image for best demo
color_defect = next((d for d in defect_types
                     if any(k in d for k in ['contamination','color','glue','stain'])),
                    first_defect)

color_folder  = os.path.join(test_folder, color_defect)
color_img_path= os.path.join(color_folder, sorted(os.listdir(color_folder))[0])
color_gt_path = os.path.join(gt_folder, color_defect,
                             os.path.basename(color_img_path).replace('.png', '_mask.png'))

img_color  = cv2.imread(color_img_path)
gt_color   = cv2.imread(color_gt_path, cv2.IMREAD_GRAYSCALE) if os.path.exists(color_gt_path) else None

mask_hsv   = segment_hsv_saturation(img_color)
iou_hsv    = compute_iou(mask_hsv, gt_color)

print(f'Defect type : {color_defect}')
print(f'Strategy    : HSV Saturation + Otsu')
print(f'IoU         : {iou_hsv:.4f}' if iou_hsv is not None else 'IoU: N/A')

# Show HSV channels
hsv = cv2.cvtColor(img_color, cv2.COLOR_BGR2HSV)
fig, axes = plt.subplots(1, 5, figsize=(25, 5))
axes[0].imshow(cv2.cvtColor(img_color, cv2.COLOR_BGR2RGB)); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(hsv[:,:,0], cmap='hsv');  axes[1].set_title('H channel'); axes[1].axis('off')
axes[2].imshow(hsv[:,:,1], cmap='gray'); axes[2].set_title('S channel (used)'); axes[2].axis('off')
axes[3].imshow(hsv[:,:,2], cmap='gray'); axes[3].set_title('V channel'); axes[3].axis('off')
axes[4].imshow(mask_hsv, cmap='hot');    axes[4].set_title(f'Predicted Mask | IoU: {iou_hsv:.4f}' if iou_hsv else 'Predicted Mask'); axes[4].axis('off')
plt.suptitle(f'Strategy 3: HSV Saturation — {color_defect}', fontsize=14)
plt.tight_layout()
plt.show()

## 5️⃣ Strategy 4: K-Means Clustering

In [ ]:
mask_km = segment_kmeans(img_defect, k=3)
iou_km  = compute_iou(mask_km, gt_mask)

print(f'Strategy : K-Means (k=3)')
print(f'IoU      : {iou_km:.4f}' if iou_km is not None else 'IoU: N/A')

# Show K=2,3,4 comparison
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(cv2.cvtColor(img_defect, cv2.COLOR_BGR2RGB))
axes[0].set_title('Original'); axes[0].axis('off')

for i, k in enumerate([2, 3, 4]):
    m = segment_kmeans(img_defect, k=k)
    iou = compute_iou(m, gt_mask)
    axes[i+1].imshow(m, cmap='hot')
    axes[i+1].set_title(f'K={k} | IoU: {iou:.4f}' if iou else f'K={k}')
    axes[i+1].axis('off')

plt.suptitle('Strategy 4: K-Means — Effect of K', fontsize=14)
plt.tight_layout()
plt.show()

## 6️⃣ All Strategies Side-by-Side Comparison

In [ ]:
compare_strategies(
    img_defect,
    gt_mask=gt_mask,
    ref_good_bgr=img_good,
    defect_type=first_defect
)

In [ ]:
# Summary bar chart
strategies  = ['Grayscale', 'Diff+Otsu', 'HSV', 'K-Means']
masks       = [mask_gray, mask_diff, mask_hsv, mask_km]
ious        = [compute_iou(m, gt_mask) or 0 for m in masks]
colors      = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']

plt.figure(figsize=(8, 5))
bars = plt.bar(strategies, ious, color=colors)
plt.ylim(0, 1.1)
plt.title(f'IoU Comparison — {CATEGORY} / {first_defect}')
plt.ylabel('IoU')
for bar, val in zip(bars, ious):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.02, f'{val:.4f}',
             ha='center', fontsize=11)
plt.tight_layout()
plt.show()

## 7️⃣ Batch Evaluation — All Defect Types

In [ ]:
print(f'\n📊 Evaluating all defect types in [{CATEGORY}]...\n')

seg_metrics = evaluate_segmentation(
    DATA_PATH,
    category=CATEGORY,
    max_images=20,
    ref_good_bgr=img_good
)

print_segmentation_metrics(seg_metrics)

In [ ]:
# ─── Plot per-type IoU bar chart ────────────────────────────
per_type = seg_metrics['per_type']
names    = list(per_type.keys())
ious_bar = [per_type[n]['mean_iou'] for n in names]

plt.figure(figsize=(max(8, len(names) * 1.5), 5))
bars = plt.bar(names, ious_bar, color='steelblue')
plt.axhline(y=seg_metrics['overall_iou'], color='red',
            linestyle='--', label=f'Overall: {seg_metrics["overall_iou"]:.4f}')
plt.xticks(rotation=45, ha='right')
plt.ylim(0, 1.1)
plt.ylabel('Mean IoU')
plt.title(f'Per-Defect-Type IoU — {CATEGORY}')
plt.legend()
for bar, val in zip(bars, ious_bar):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.01, f'{val:.3f}',
             ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## 8️⃣ Save Segmentation Outputs

In [ ]:
# ─── Load test image paths ──────────────────────────────────
image_paths = load_image_paths(DATA_PATH, category=CATEGORY, split='test', max_images=60)
print(f'Test images loaded: {len(image_paths)}')

output_dir  = os.path.join('outputs', 'segmentation', CATEGORY)

result = batch_segment(
    image_paths,
    save_samples=True,
    sample_limit=5,
    output_dir=output_dir,
    ref_good_bgr=img_good,
    category=CATEGORY
)

print(f'\n=== Batch Segmentation Result ===')
print(f'Mean IoU : {result["mean_iou"]:.4f}')
print(f'Images   : {result["n"]}')
print(f'Outputs  : {output_dir}')

In [ ]:
# ─── Visualize saved outputs ────────────────────────────────
if os.path.exists(output_dir):
    sample_files = sorted([f for f in os.listdir(output_dir) if 'overlay' in f])
    n = min(len(sample_files), 5)

    if n > 0:
        fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
        if n == 1:
            axes = [axes]
        for ax, fname in zip(axes, sample_files[:n]):
            img = cv2.imread(os.path.join(output_dir, fname))
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            ax.set_title(fname)
            ax.axis('off')
        plt.suptitle('Sample Segmentation Overlays', fontsize=14)
        plt.tight_layout()
        plt.show()

## 9️⃣ Multi-Category Evaluation
Run on multiple categories to get a comprehensive results table.

In [ ]:
# ─── Change this list to evaluate more categories ───────────
EVAL_CATEGORIES = ['bottle', 'leather', 'hazelnut']

all_results = {}

for cat in EVAL_CATEGORIES:
    print(f'\n{"-"*50}')
    print(f'  Evaluating: {cat.upper()}')
    print(f'{"-"*50}')

    # Load reference good image
    good_folder = os.path.join(DATA_PATH, cat, 'test', 'good')
    ref_path    = os.path.join(good_folder, sorted(os.listdir(good_folder))[0])
    ref_bgr     = cv2.imread(ref_path)

    metrics = evaluate_segmentation(
        DATA_PATH, category=cat, max_images=10, ref_good_bgr=ref_bgr
    )
    all_results[cat] = metrics['overall_iou']

# Print summary table
print('\n' + '='*40)
print(f'{"Category":15s} | {"Overall IoU":12s}')
print('-'*40)
for cat, iou in all_results.items():
    print(f'{cat:15s} | {iou:.4f}')
print('='*40)